In [7]:
# 1. 최신 transformers (SigLIP2용, Jina와 반대)
!pip install -q "transformers>=4.50" peft accelerate
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [9]:
"""
SigLIP2 LoRA 파인튜닝 (LookMatch) — Jina-CLIP 비교용
=====================================================
crop된 옷 이미지 + qwen_caption 쌍으로 SigLIP2를 도메인 파인튜닝.
Jina-CLIP v2와 같은 데이터·조건으로 학습해 성능 비교.

Jina와 차이:
  - SigLIP2는 표준 구조 → FlashAttention 강제 없음
  - 텍스트+비전 둘 다 LoRA 학습 가능 (Jina는 비전만이었음)
  - get_image_features / get_text_features API (Jina와 유사)

모델: google/siglip2-so400m-patch16-512 (512, Jina와 급 비슷)
  더 가볍게: google/siglip2-base-patch16-512

입력:  metadata_captioned_final.csv (crop_path, qwen_caption)
출력:  siglip2_finetuned/  (LoRA 가중치)

설치:
  !pip install -q "transformers>=4.50" peft pillow pandas tqdm accelerate

실행:  python finetune_siglip2.py
  ※ SigLIP2는 최신 transformers 필요 (Jina와 반대 — 주의!)
"""

import os
import gc
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm


# ============================================================
# 경로 설정 — 로컬 / Colab / 드라이브 자동 감지
# ============================================================
def setup_data_dir():
    env_dir = os.environ.get("LOOKMATCH_DIR")
    if env_dir:
        return Path(env_dir), False
    try:
        import google.colab  # noqa
        drive_root = Path("/content/drive/MyDrive")
        if not drive_root.exists():
            from google.colab import drive
            drive.mount("/content/drive")
        data_dir = drive_root / "clothes_dataset"
        data_dir.mkdir(parents=True, exist_ok=True)
        print(f"데이터 경로(구글드라이브): {data_dir}")
        return data_dir, True
    except ImportError:
        data_dir = Path("clothes_dataset")
        print(f"데이터 경로(로컬): {data_dir.resolve()}")
        return data_dir, False


DATA_DIR, IS_COLAB = setup_data_dir()

IN_CSV = DATA_DIR / "metadata_captioned_final.csv"
if not IN_CSV.exists():
    for alt in ["metadata_captioned.csv", "metadata_captioned_test_final.csv"]:
        if (DATA_DIR / alt).exists():
            IN_CSV = DATA_DIR / alt
            break

OUT_DIR = str(DATA_DIR / "siglip2_finetuned")
CROPS_SUBDIR = "crops"
LOCAL_CROPS = Path("/content/crops_local") if IS_COLAB else (DATA_DIR / CROPS_SUBDIR)

# ============================================================
# 하이퍼파라미터
# ============================================================
# 512 해상도 (현욱님 패딩 이미지에 맞음). Jina와 급 맞추려면 so400m
MODEL_NAME = "google/siglip2-so400m-patch16-512"
# 더 가볍게 하려면: "google/siglip2-base-patch16-512"
BATCH_SIZE = 8           # SigLIP2 512, LoRA면 8 가능 (OOM 시 4)
EPOCHS = 3
LR = 1e-5
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
VAL_RATIO = 0.1
SEED = 42
SMOKE_TEST = False        # True면 200개로 검증
SMOKE_N = 200
SKIP_LOCAL_COPY = True   # 드라이브 직접 읽기 (복사 생략)
# 텍스트도 학습할지: SigLIP2는 표준이라 True 가능 (Jina는 불가였음)
TRAIN_TEXT = True        # True면 텍스트+비전, False면 비전만(Jina와 같은 조건)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


def resolve_image_path(crop_path):
    fname = str(crop_path).replace("\\", "/").split("/")[-1]
    for cand in [LOCAL_CROPS / fname, Path(crop_path), DATA_DIR / CROPS_SUBDIR / fname]:
        if Path(cand).exists():
            return str(cand)
    return None


def copy_images_local(df):
    if not IS_COLAB or SKIP_LOCAL_COPY:
        if SKIP_LOCAL_COPY:
            print("SKIP_LOCAL_COPY=True → 드라이브에서 직접 읽습니다.")
        return
    # (생략: 필요 시 Jina 코드의 병렬 복사 로직 재사용)


# ============================================================
# 데이터셋
# ============================================================
class ClothesDataset(Dataset):
    def __init__(self, df):
        self.rows = []
        for _, row in df.iterrows():
            img = resolve_image_path(row["crop_path"])
            cap = str(row.get("qwen_caption", "")).strip()
            if img and cap:
                self.rows.append((img, cap))
        print(f"  유효 샘플: {len(self.rows)}개")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        img_path, caption = self.rows[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (512, 512), (255, 255, 255))
        return {"image": image, "caption": caption}


def collate_fn(batch):
    return {"images": [b["image"] for b in batch],
            "captions": [b["caption"] for b in batch]}


# ============================================================
# 모델 로드 + LoRA
# ============================================================
def load_model():
    from transformers import AutoModel
    from peft import LoraConfig, get_peft_model

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"디바이스: {device}")
    print(f"모델 로드: {MODEL_NAME}")

    dtype = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
    model = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(device)

    # 전체 freeze 후 LoRA만 학습
    for p in model.parameters():
        p.requires_grad = False

    # LoRA 대상: 비전 + (옵션)텍스트의 attention/mlp Linear
    targets = []
    for name, module in model.named_modules():
        if not isinstance(module, torch.nn.Linear):
            continue
        is_vision = "vision_model" in name
        is_text = "text_model" in name
        # 텍스트 학습 여부에 따라 대상 결정
        if is_vision or (TRAIN_TEXT and is_text):
            if any(name.endswith(s) for s in
                   ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]):
                targets.append(name)
    print(f"LoRA 대상 모듈: {len(targets)}개 (텍스트 학습={TRAIN_TEXT})")

    lora_cfg = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=targets, bias="none",
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model, device


def get_base(model):
    if hasattr(model, "base_model") and hasattr(model.base_model, "model"):
        return model.base_model.model
    return model


# ============================================================
# 임베딩 (gradient 흐름) — SigLIP2 표준 API
# ============================================================
_PROC = {}
def _get_processor():
    if "p" not in _PROC:
        from transformers import AutoProcessor
        _PROC["p"] = AutoProcessor.from_pretrained(MODEL_NAME)
    return _PROC["p"]


def _to_tensor(out):
    """get_*_features 반환이 텐서면 그대로, 객체면 임베딩 텐서 추출"""
    if torch.is_tensor(out):
        return out
    for attr in ["image_embeds", "text_embeds", "pooler_output", "last_hidden_state"]:
        if hasattr(out, attr):
            v = getattr(out, attr)
            if torch.is_tensor(v):
                if attr == "last_hidden_state" and v.dim() == 3:
                    return v.mean(dim=1)
                return v
    raise TypeError(f"임베딩 텐서를 찾을 수 없음: {type(out)}")


def encode_batch(model, images, captions, device):
    base = get_base(model)
    proc = _get_processor()
    # SigLIP2: 이미지/텍스트 각각 처리
    img_inputs = proc(images=images, return_tensors="pt").to(device)
    txt_inputs = proc(text=captions, return_tensors="pt",
                      padding="max_length", max_length=64, truncation=True).to(device)
    img_emb = _to_tensor(base.get_image_features(**img_inputs))
    txt_emb = _to_tensor(base.get_text_features(**txt_inputs))
    return img_emb.float(), txt_emb.float()


def clip_loss(img_emb, txt_emb, temperature=0.07):
    img_emb = F.normalize(img_emb, dim=-1)
    txt_emb = F.normalize(txt_emb, dim=-1)
    logits = (img_emb @ txt_emb.t()) / temperature
    labels = torch.arange(len(img_emb), device=img_emb.device)
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.t(), labels)) / 2


@torch.no_grad()
def evaluate(model, loader, device, k=5):
    model.eval()
    all_i, all_t = [], []
    for b in loader:
        ie, te = encode_batch(model, b["images"], b["captions"], device)
        all_i.append(F.normalize(ie, dim=-1).cpu())
        all_t.append(F.normalize(te, dim=-1).cpu())
        del ie, te
        if device == "cuda":
            torch.cuda.empty_cache()
    if not all_i:
        return 0.0
    ie, te = torch.cat(all_i), torch.cat(all_t)
    sims = te @ ie.t()
    topk = sims.topk(min(k, sims.size(1)), dim=1).indices
    correct = sum(i in topk[i] for i in range(len(topk)))
    return correct / len(topk)


# ============================================================
# 학습
# ============================================================
def train():
    print(f"입력 CSV: {IN_CSV}")
    if not IN_CSV.exists():
        print("[오류] 입력 CSV 없음.")
        return

    df = pd.read_csv(IN_CSV)
    df = df[df["qwen_caption"].notna() & (df["qwen_caption"].astype(str).str.strip() != "")]
    if SMOKE_TEST:
        df = df.head(SMOKE_N).copy()
        print(f"[SMOKE TEST] {len(df)}개")
    print(f"학습 대상: {len(df)}개")

    copy_images_local(df)

    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n_val = max(1, int(len(df) * VAL_RATIO))
    val_df, train_df = df[:n_val], df[n_val:]
    print("학습셋:", end=" "); train_ds = ClothesDataset(train_df)
    print("검증셋:", end=" "); val_ds = ClothesDataset(val_df)

    n_workers = 8 if (IS_COLAB and SKIP_LOCAL_COPY) else 2
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, num_workers=n_workers)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=collate_fn, num_workers=n_workers)

    model, device = load_model()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=LR)

    print(f"\n학습 시작 (epochs={EPOCHS}, batch={BATCH_SIZE})\n")
    best = 0.0
    for epoch in range(EPOCHS):
        model.train()
        total = 0.0
        t0 = time.time()
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for b in pbar:
            try:
                optimizer.zero_grad()
                ie, te = encode_batch(model, b["images"], b["captions"], device)
                loss = clip_loss(ie, te)
                if torch.isnan(loss) or torch.isinf(loss):
                    optimizer.zero_grad(); continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0)
                optimizer.step()
                total += loss.item()
                pbar.set_postfix(loss=f"{loss.item():.4f}")
                del ie, te, loss
            except torch.cuda.OutOfMemoryError:
                print(f"\n[OOM] 배치 건너뜀 (BATCH_SIZE↓ 권장)")
                optimizer.zero_grad()
                torch.cuda.empty_cache() if device=="cuda" else None
                continue

        recall = evaluate(model, val_loader, device, k=5)
        dt = time.time() - t0
        print(f"  Epoch {epoch+1}: loss={total/max(len(train_loader),1):.4f}, "
              f"Recall@5={recall:.3f}, {dt:.0f}초")

        if recall >= best:
            best = recall
            Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
            try:
                model.save_pretrained(OUT_DIR)
                print(f"  → 저장: {OUT_DIR}")
            except Exception as e:
                print(f"  저장 경고: {e}")
        gc.collect()
        torch.cuda.empty_cache() if device=="cuda" else None

    print(f"\n{'='*55}")
    print(f"학습 완료! 최고 Recall@5 = {best:.3f}")
    print(f"LoRA 가중치: {OUT_DIR}")
    if SMOKE_TEST:
        print("※ SMOKE 완료 → SMOKE_TEST=False로 전체 학습")
    print(f"{'='*55}")


if __name__ == "__main__":
    train()

데이터 경로(구글드라이브): /content/drive/MyDrive/clothes_dataset
입력 CSV: /content/drive/MyDrive/clothes_dataset/metadata_captioned_final.csv
학습 대상: 71637개
SKIP_LOCAL_COPY=True → 드라이브에서 직접 읽습니다.
학습셋:   유효 샘플: 64474개
검증셋:   유효 샘플: 7163개
디바이스: cuda
모델 로드: google/siglip2-so400m-patch16-512


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

LoRA 대상 모듈: 327개 (텍스트 학습=True)
trainable params: 17,602,048 || all params: 1,154,157,746 || trainable%: 1.5251

학습 시작 (epochs=3, batch=8)



Epoch 1/3: 100%|██████████| 8060/8060 [1:28:40<00:00,  1.51it/s, loss=0.0000]  


  Epoch 1: loss=0.0526, Recall@5=0.555, 5490초
  → 저장: /content/drive/MyDrive/clothes_dataset/siglip2_finetuned


Epoch 2/3: 100%|██████████| 8060/8060 [1:06:47<00:00,  2.01it/s, loss=0.0000]


  Epoch 2: loss=0.0322, Recall@5=0.582, 4181초
  → 저장: /content/drive/MyDrive/clothes_dataset/siglip2_finetuned


Epoch 3/3: 100%|██████████| 8060/8060 [1:08:04<00:00,  1.97it/s, loss=0.0000]


  Epoch 3: loss=0.0289, Recall@5=0.584, 4262초
  → 저장: /content/drive/MyDrive/clothes_dataset/siglip2_finetuned

학습 완료! 최고 Recall@5 = 0.584
LoRA 가중치: /content/drive/MyDrive/clothes_dataset/siglip2_finetuned
